Notebook to practice to construct a nice TPM data processing pipeline.

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import sys

In [2]:
fcnts_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/pipeline_practice/Fcnts"

# Store folder name
files = os.listdir(fcnts_path)

# Filter out the summary files and keep only NDC comparisons
files = [csv for csv in files if ".summary" not in csv and "NDC0hr" in csv]

# Attach the folder to each of the files
files = ["".join([fcnts_path, "/" , csv]) for csv in files]

# Load each file as a dataframe
dfs = [pd.read_table(csv, sep = "\t", header = 0, skiprows = 1) for csv in files]

# Rename the columns of each pandas dataframe




In [7]:
practice_df = dfs[0]
practice_df.head(n = 5)



# Subset to the columns that either start with a "/" or are named "Genedid"





# Remember, we only need to get the NDC0hr columns from the 1st thing??
# Lowkey just process everything the same, then remove duplicate columns at the end to get rid of all the NDC?





# Set Gene names to the row names
practice_df = practice_df.set_index("Geneid")

# Select length column and those that correspond to the Fcnts values
practice_df = practice_df.loc[:, [col for col in list(practice_df.columns) if col == "Length" or col.startswith("/")]]

# Convert every entry of the dataframe to an integer to allow for integer manipulation
practice_df = practice_df.apply(lambda column: [int(entry) for entry in column])

# Divide Length by 1000 to get kb
practice_df["Length"] = practice_df["Length"].apply(lambda column: column / 1000)

# Select only columns with Fcnts
fcnts_cols = [col for col in list(practice_df.columns) if col != "Length"]

# Fcnts / gene length = (Counts per kb)
practice_df[fcnts_cols] = practice_df[fcnts_cols].apply(lambda column: column / practice_df["Length"])

# (Counts per kb) * 10^6 / (Total counts/kb) = TPM
practice_df[fcnts_cols] = practice_df[fcnts_cols].apply(lambda column: column * 10**6 / sum(column))

# Remove length column
practice_df.drop(columns = "Length", inplace = True)

# Rename the columns to remove extra crap

# can use df.rename(lambda x: ) to rename the samples?

# Obtain index of the last instance of / in the name of the thing
# Remove ".bam"
def sample_name_strip(name):
    """
    
    """
    # Find index of the last / 
    samplename_start_idx = name.strip().rfind("/") + 1
    
    # Subset 
    new_name = name[samplename_start_idx:]

    # Find index of . (.bam is at end of sample name) and remove filetag
    filetag_start_idx = new_name.rfind(".")
    new_name = new_name[:filetag_start_idx]
    
    # Remove "T4-wt"
    new_name = new_name.replace("T4-wt", "")

    return new_name

# Apply stripper to each column names
practice_df = practice_df.rename(columns = lambda column: sample_name_strip(column)).T

In [6]:
cfu_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/pipeline_practice/all_cfus"

# Store folder name
cfu_files = os.listdir(cfu_path)

# Attach the folder to each of the files
cfu_files = ["".join([cfu_path, "/" , csv]) for csv in cfu_files]

# Load each file as a dataframe
cfu_dfs = [pd.read_table(csv, sep = ",", header = 0) for csv in cfu_files]

# For each df
for i,df in enumerate(cfu_dfs):

    # Melt to get all triplicates stacked
    df = df.melt(id_vars = "Triplicates", var_name = "Condition", value_name = "CFU")

    # Define labels by combining condition + "-" + triplicate label
    df["Labels"] = [df["Condition"][i].strip() + "-" + df["Triplicates"][i].strip() for i in range(df.shape[0])]

    # Drop unncessary columns
    df = df.drop(columns = ["Condition", "Triplicates"])

    # Move labels to index
    df = df.set_index("Labels")
    cfu_dfs[i] = df

pd.concat(cfu_dfs)

,CFU
Labels,
NDC1hr-a,580000000
NDC1hr-b,1000000000
NDC1hr-c,800000000
14CEF1hr-a,340000000
14CEF1hr-b,700000000
...,...
34CIP12VNC4hr-b,2200000
34CIP12VNC4hr-c,5000000
34CIP34VNC4hr-a,1800000


In [8]:
practice_df

Geneid,SP_0001,SP_0002,SP_0003,SP_0004,SP_0005,SP_0006,SP_0007,SP_0008,SP_0009,SP_0010,...,SP_2230,SP_2231,SP_2233,SP_2234,SP_2235,SP_2236,SP_2237,SP_2238,SP_2239,SP_2240
12CEF12CIP1hr-a,114.547926,73.295354,50.592072,161.505461,34.001150,27.607771,68.957388,86.661467,47.269891,64.144013,...,298.531351,78.570224,0.000000,24.984270,24.451026,24.846994,7.690736,24.225819,2450.455951,2051.693877
12CEF12CIP1hr-b,109.150760,102.588865,88.178438,149.196178,39.322556,28.670553,41.116912,115.286138,18.594538,71.371377,...,324.679202,81.702348,5.956063,32.011371,20.046542,18.283227,0.000000,41.930684,2475.206794,2085.234170
12CEF12CIP1hr-c,113.862974,84.181428,72.986338,190.529657,37.728815,28.045676,61.259319,117.381415,24.625472,71.844668,...,326.221497,74.388602,5.915885,15.060993,17.698945,19.416237,14.423491,33.444469,2671.713668,1988.560367
NDC0hr-a,102.614861,83.732058,54.760090,170.615323,26.057362,26.221659,60.187756,87.100980,17.816110,60.056247,...,140.721286,49.498498,14.583847,74.436121,17.784563,23.136758,1.932435,14.203398,1123.912135,1292.501235
NDC0hr-b,108.551217,105.322135,46.243152,182.004097,35.553683,33.067411,51.750764,84.876862,14.978270,76.582159,...,111.243416,51.055483,10.794886,65.312704,19.573197,23.966926,0.000000,28.786362,1092.615364,1319.242197
NDC0hr-c,95.498504,93.891570,42.709114,179.954974,30.138456,27.967309,39.829848,84.242992,16.626906,71.876976,...,134.324505,46.141131,14.912257,104.703579,21.184401,31.771686,6.492411,21.303224,1137.094714,1358.556260


In [64]:
# All functions for data conversion and extraction from specified path
import pandas as pd
import numpy as np
import os
import functools

from functools import reduce

def read_fcnts_as_df(folder_path):
    """
    Extracts all feature count csvs from a specified folder and stores them as a list of dataframes

    Args:
        folder_path : File path containing Fcnts csv output from pipeline

    Returns: 
        fcnt_df_list : List of Fcnts dataframes
    """
    
    # Filter out the summary files and keep Fcnts from no-drug control (NDC) comparisons
    all_files = os.listdir(folder_path)
    files = [
        f for f in all_files
        if f.endswith(".csv")
        and ".summary" not in f
        and "NDC0hr" in f
            ]

    if len(files) == 0: 
        print("Directory path is empty")
        return

    # Convert each csv (which is stored as tab-delimited) to dataframe
    filenames = ["".join([folder_path, "/" , f]) for f in files]
    fcnt_df_list = [pd.read_table(f, sep = "\t", header = 0, skiprows = 1) for f in filenames]

    return fcnt_df_list

def sample_name_strip(name):
            """
            Convert a sample file name into an easy to read sample name
            Args:
                name : (ex: "/ExpOut/260107_AV242502_RNASeq_miniHT_SpnT4WT_CEF_CIP/Out/Rep/Bams/T4-wt12CEF12CIP1hr-a.bam")
            
            Returns:
                new_name : (ex: 12CEF12CIP1hr-a)
            """
            # Find index of the last / and remove entire prefix (OG file path)
            samplename_start_idx = name.strip().rfind("/") + 1
            new_name = name[samplename_start_idx:]

            # Find index of . (.bam is at end of sample name) and remove filetag
            filetag_start_idx = new_name.rfind(".")
            new_name = new_name[:filetag_start_idx]
    
            # Remove "T4-wt"
            new_name = new_name.replace("T4-wt", "")
    
            return new_name


def fcnts_to_tpms(fcnt_df_list):
    """
    Converts a list of RNA-seq feature count dataframes to a list of TPM dataframes

    Args:
        fcnt_df_list : List of feature count dataframes

    Returns:
        tpm_df_list : List of TPM dataframes

    """
    tpm_df_list = []

    for df in fcnt_df_list:

        # Move gene names to index and keep only length, Fcnts
        df = df.set_index("Geneid")
        df = df.loc[:, [col for col in list(df.columns) if col == "Length" or col.startswith("/")]]
        df = df.apply(lambda column: [int(entry) for entry in column])

        # Convert gene length from b -> kb
        df["Length"] = df["Length"].apply(lambda column: column / 1000)

        # Select Fcnts columns by excluding Length column
        fcnts_cols = [col for col in list(df.columns) if col != "Length"]
        
        # Fcnts / gene length = (Counts per kb)
        df[fcnts_cols] = df[fcnts_cols].apply(lambda column: column / df["Length"])

        # (Counts per kb) * 10^6 / (Total counts/kb) = TPM
        df[fcnts_cols] = df[fcnts_cols].apply(lambda column: column * 10**6 / sum(column))

        # Remove length column 
        df.drop(columns = "Length", inplace = True)
        
        # Apply stripper to each column names
        df = df.rename(columns = lambda column: sample_name_strip(column))
        tpm_df_list.append(df)        

    return tpm_df_list

def bind_tpm_data(tpm_df_list):
    """
    Function to take a list of TPM dataframes, then bind all into 1 dataframe
    Args: 
        tpm_df_list : list of TPM dataframes
    Output:
        all_tpms [N,G] : dataframe with all TPM values (N samples on row, G genes on column)
    """
    tpm_df_list_uniq = []

    # Column names of 1st DF (all have last 3 cols)
    colnames = list(tpm_df_list[0].columns)

    # Select redundant NDC0hr columns and make new df with just those
    ndc0hr_idx = [i for i in range(len(colnames)) if "NDC0hr" in colnames[i]]
    ndc0hr_df = tpm_df_list[0].iloc[:,ndc0hr_idx]
    tpm_df_list_uniq.append(ndc0hr_df)

    # Remove NDC0hr columns from all dfs
    for df in tpm_df_list:
        columns = list(tpm_df_list[0].columns)
        relevant_idx = [i for i in range(len(colnames)) if "NDC0hr" not in colnames[i]]
        stripped_df = tpm_df_list[0].iloc[:,relevant_idx]
        tpm_df_list_uniq.append(stripped_df)

    # Iterated outer join by index
    all_tpms = reduce(lambda df1, df2 :
                      pd.merge(df1, df2, # Only take nonoverlapping cols
                               left_index = True, 
                               right_index = True, 
                               how = "outer"),
                      tpm_df_list_uniq)

    # Tranpose to get genes on columns
    all_tpms = all_tpms.T

    return all_tpms

# Function to load the CFU data
def read_cfus(folder_path):
    """
    Function to extract CFUs into a dataframe with conditions on rows, 1 CFU column
    Args:
        folder_path : path to folder containing CFUs (same format as /all_cfus)
    Output:
        all_cfus [N,1] : df with condition names as index, 1 column of CFUs (N = # samples)
    """

    # Get files
    files = os.listdir(folder_path)
    
    # Select CSV files
    cfu_files = [csv for csv in files if ".csv" in csv]

    # Join path
    cfu_files = ["".join([folder_path, "/", csv] for csv in files)]
    
    # Load each file as a dataframe
    cfu_dfs = [pd.read_table(csv, sep = ",", header = 0) for csv in cfu_files] 

    for i, df in enumerate(cfu_dfs):

        # Melt to get all triplicates stacked
        df = df.melt(id_vars = "Triplicates", var_name = "Condition", value_name = "CFU")

        # Define labels by combining condition + "-" + triplicate label
        df["Labels"] = [df["Condition"][i].strip() + "-" + df["Triplicates"][i].strip() for i in range(df.shape[0])]

        # Drop unncessary columns
        df = df.drop(columns = ["Condition", "Triplicates"])

        # Move labels to index
        df = df.set_index("Labels")
        cfu_dfs[i] = df
    
    # Concat
    all_cfus = pd.concat(cfu_dfs)

    return all_cfus

# Function to bind TPMs
def bind_all_data(tpm_df, cfu_df, outfig):
    """
    Function bind TPM and cfu dfs
    Args:
        tpm_df [N,G] : Dataframe of TPMs, N = # samples, G = # genes, labels on index
        cfu_df [N,1] : Dataframe of CFUs, labels on index
    Output:
        data_df : [N, G+1] : Dataframe of all TPMs and CFUs as last column
    """
    # Right join so that CFUs exist
    data_df = pd.merge(tpm_df, cfu_df, left_index = True, right_index = True, how = "right")

    return data_df

def data_extract(fcnts_path, cfu_path):
    """
    Function to run entire data extraction pipeline

    Args:
        fcnts_path : Path to folder containing Fcnts output of lab pipeline
        cfu_path   : Path to folder containing CFU csv outputs
    Returns:
        all_data : [N, G+1] : Dataframe of all TPMs and CFUs as last column 
    """
    stored_fcnts = read_fcnts_as_df(fcnts_path)
    stored_tpms  = fcnts_to_tpms(stored_fcnts)
    tpm_df       = bind_tpm_data(stored_tpms)
    cfu_df       = read_cfus(cfu_path)
    all_data     = bind_all_data(tpm_df, cfu_df)
    
    return all_data

In [65]:
fcnts_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/fcnts_timezero"

cfu_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/all_cfus"

ex = fcnts_to_tpms(read_fcnts_as_df(fcnts_path))

In [66]:
bind_tpm_data(ex)

MergeError: Passing 'suffixes' which cause duplicate columns {'12AMX1hr-b_x', '12AMX1hr-a_x', '12AMX1hr-c_x'} is not allowed.